# 04 — ML Improvement Experiments: Rare-Class Analysis, Robustness and Interpretability

Notebook 03 selected a KNN pipeline by group-aware cross-validation and evaluated it once on the held-out
test set. It reaches very high accuracy but recovers only a small fraction of the rare `reset-both` class.
This notebook investigates why and documents the trade-offs between alternatives.

**Rules:** training data only (`test.csv` is never loaded); same folds as Notebook 03; the KNN uses the deployed
hyperparameters from `models/metadata.json`; Notebook 03 is left unchanged; the overlap audit and strict group
CV already in Notebook 03 are cited, not repeated.

**Questions:** (1) precision/recall trade-off of the six classifiers on `reset-both`; (2) effect of class
weights; (3) where the missed rows go; (4) can a decision rule recover them; (5) which features carry the signal
and which exist only after the decision; (6) should the deployed model change?

## 1. Setup
Switches: `N_REPEATS` (permutation repeats), `RUN_EXTENSION` (extra notebook-local scenario without
`Elapsed Time (sec)`), `UPDATE_METRICS_JSON` (keep False until results are reviewed).

In [8]:
import sys, time, json, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():          # allow running from notebooks/
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Run from the repository root or from notebooks/."
sys.path.insert(0, str(ROOT))

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix,
                             balanced_accuracy_score, f1_score)
from sklearn.inspection import permutation_importance

import src.preprocessing as prep
from src.models import build_model, load_metadata, CLASS_LABELS, RANDOM_STATE
from src.preprocessing import get_feature_columns

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

N_REPEATS = 3
RUN_EXTENSION = True
UPDATE_METRICS_JSON = False

print("python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)
print("repository root:", ROOT.name)

python 3.14.4 | scikit-learn 1.9.1 | pandas 3.1.0 | numpy 2.5.4
repository root: Firewall-Traffic-Analyzer


## 2. Data, grouped folds and the deployed configuration
Group id = hash of the full 11-feature row (same as Notebook 03). `reset-both` has 43 training rows, so each
validation fold has 8–9; every fold must contain it. Sanity check: the mean macro-F1 must reproduce the `full`
row of `reports/feature_ablation.csv`; otherwise the folds differ and nothing below is comparable.

In [9]:
train = pd.read_csv(ROOT / "data" / "processed" / "train.csv")   # test.csv is intentionally NOT loaded
FULL = get_feature_columns("full")
X_train, y_train = train[FULL], train["Action"]
assert len(FULL) == 11 and set(train.columns) == set(FULL) | {"Action"}

groups = pd.util.hash_pandas_object(X_train, index=False).to_numpy()
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train, groups=groups))

for i, (tr, va) in enumerate(folds, 1):
    assert set(groups[tr]).isdisjoint(set(groups[va])), "group leakage between train/validation"
    counts = y_train.iloc[va].value_counts().reindex(CLASS_LABELS, fill_value=0).to_dict()
    print(f"fold {i}: validation rows={len(va):>5} | {counts}")
    assert counts["reset-both"] > 0, "a fold without reset-both"

meta = load_metadata()
DEPLOYED_PARAMS = meta["best_params"]
print("\ndeployed model:", meta["model_name"], "| feature_set:", meta["feature_set"], "|", DEPLOYED_PARAMS)

def deployed_knn(scenario="full"):
    model = build_model("KNN", scenario)
    model.set_params(**DEPLOYED_PARAMS)
    return model

def oof_predict(model, cols, method="predict"):
    """Out-of-fold predictions: every training row is predicted by a model that never saw it."""
    return cross_val_predict(model, X_train[cols], y_train, cv=folds, n_jobs=1, method=method)

def fold_means(pred):
    """Mean/std of per-fold macro-F1 and mean balanced accuracy (the Notebook-03 protocol)."""
    f1s, bas = [], []
    for _, va in folds:
        yt, yp = y_train.iloc[va], pred[va]
        f1s.append(f1_score(yt, yp, labels=CLASS_LABELS, average="macro", zero_division=0))
        bas.append(balanced_accuracy_score(yt, yp))
    return float(np.mean(f1s)), float(np.std(f1s)), float(np.mean(bas))

ref = pd.read_csv(REPORTS / "feature_ablation.csv").set_index("feature_set")
pred_deployed = oof_predict(deployed_knn(), FULL)
m_f1, s_f1, m_ba = fold_means(pred_deployed)
print(f"\nmean fold macro-F1 here = {m_f1:.6f} | Notebook 03 = {ref.loc['full', 'f1_macro_mean']:.6f}")
assert abs(m_f1 - ref.loc["full", "f1_macro_mean"]) < 0.002, "folds differ from Notebook 03"

fold 1: validation rows= 9148 | {'allow': 5990, 'deny': 1287, 'drop': 1862, 'reset-both': 9}
fold 2: validation rows= 9147 | {'allow': 5990, 'deny': 1287, 'drop': 1862, 'reset-both': 8}
fold 3: validation rows= 9147 | {'allow': 5990, 'deny': 1286, 'drop': 1862, 'reset-both': 9}
fold 4: validation rows= 9147 | {'allow': 5991, 'deny': 1287, 'drop': 1861, 'reset-both': 8}
fold 5: validation rows= 9147 | {'allow': 5990, 'deny': 1287, 'drop': 1861, 'reset-both': 9}

deployed model: KNN | feature_set: full | {'classifier__n_neighbors': 21, 'classifier__weights': 'distance'}

mean fold macro-F1 here = 0.847813 | Notebook 03 = 0.847813


## 3. Out-of-fold analysis of the rare class `reset-both`
Precision = of the rows predicted `reset-both`, how many are real. Recall = of the real rows, how many were found.
High recall with terrible precision is not automatically better. `pooled_macro_f1` is computed once on all
out-of-fold predictions; `mean_fold_macro_f1` is the Notebook-03 protocol: never mix them in one column.
`misses_predicted_deny` counts real `reset-both` rows labelled `deny`. Out-of-fold, training set only.

In [10]:
def rare_row(name, pred):
    rep = classification_report(y_train, pred, labels=CLASS_LABELS, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_train, pred, labels=CLASS_LABELS)
    r = CLASS_LABELS.index("reset-both"); d = CLASS_LABELS.index("deny")
    mean_f1, _, _ = fold_means(pred)
    return dict(
        model=name,
        rare_precision=rep["reset-both"]["precision"],
        rare_recall=rep["reset-both"]["recall"],
        rare_f1=rep["reset-both"]["f1-score"],
        rare_tp=int(cm[r, r]),
        rare_fn=int(cm[r].sum() - cm[r, r]),
        rare_fp=int(cm[:, r].sum() - cm[r, r]),
        misses_predicted_deny=int(cm[r, d]),
        pooled_macro_f1=rep["macro avg"]["f1-score"],
        mean_fold_macro_f1=mean_f1,
        balanced_accuracy=balanced_accuracy_score(y_train, pred),
    )

def with_weight(name, weight):
    model = build_model(name, "full")
    model.set_params(classifier__class_weight=weight)
    return model

CANDIDATES = {
    "KNN (deployed)":                      deployed_knn(),
    "Decision Tree (balanced)":            build_model("Decision Tree", "full"),
    "Random Forest (balanced_subsample)":  build_model("Random Forest", "full"),
    "Linear SVM (balanced)":               build_model("Linear SVM", "full"),
    "Gaussian Naive Bayes":                build_model("Gaussian Naive Bayes", "full"),
    "Decision Tree (unweighted)":          with_weight("Decision Tree", None),
    "Random Forest (unweighted)":          with_weight("Random Forest", None),
}

oof, rows = {"KNN (deployed)": pred_deployed}, []
for name, model in CANDIDATES.items():
    t0 = time.perf_counter()
    if name not in oof:
        oof[name] = oof_predict(model, FULL)
    row = rare_row(name, oof[name]); row["seconds"] = round(time.perf_counter() - t0, 1)
    rows.append(row); print(f"done: {name:<38} {row['seconds']:>6.1f}s")

all_df = pd.DataFrame(rows)
default_names = ["KNN (deployed)", "Decision Tree (balanced)", "Random Forest (balanced_subsample)",
                "Linear SVM (balanced)", "Gaussian Naive Bayes"]
minority_df = all_df[all_df["model"].isin(default_names)].reset_index(drop=True)
minority_df.to_csv(REPORTS / "minority_class_comparison.csv", index=False)
display(minority_df.round(4))

done: KNN (deployed)                            0.6s
done: Decision Tree (balanced)                  1.2s
done: Random Forest (balanced_subsample)        4.0s
done: Linear SVM (balanced)                     1.7s
done: Gaussian Naive Bayes                      1.0s
done: Decision Tree (unweighted)                1.1s
done: Random Forest (unweighted)                2.9s


,model,rare_precision,rare_recall,rare_f1,rare_tp,rare_fn,rare_fp,misses_predicted_deny,pooled_macro_f1,mean_fold_macro_f1,balanced_accuracy,seconds
0,KNN (deployed),1.0000,0.2558,0.4074,11,32,0,32,0.8500,0.8478,0.8126,0.6
1,Decision Tree (balanced),0.0376,0.9070,0.0723,39,4,997,4,0.7455,0.7455,0.9366,1.2
2,Random Forest (balanced_subsample),0.0380,0.8837,0.0729,38,5,962,5,0.7465,0.7464,0.9322,4.0
3,Linear SVM (balanced),0.2391,0.2558,0.2472,11,32,35,32,0.8091,0.8113,0.8112,1.7
4,Gaussian Naive Bayes,0.0184,1.0000,0.0361,43,0,2293,0,0.7074,0.7074,0.9147,1.0


## 4. Effect of class weights
`class_weight="balanced"` makes every class count equally in the loss, so the model predicts the rare class more
often. We compare each weighted tree model with an unweighted copy (same hyperparameters). Resampling (SMOTE)
is deliberately not used: it would have to run inside each training fold. Weighting usually raises recall and
lowers precision; we record both.

In [11]:
pairs = [("Decision Tree", "Decision Tree (balanced)", "Decision Tree (unweighted)"),
         ("Random Forest", "Random Forest (balanced_subsample)", "Random Forest (unweighted)")]
weight_rows = []
for family, weighted, unweighted in pairs:
    for label, name in [("balanced", weighted), ("none", unweighted)]:
        r = all_df.set_index("model").loc[name].to_dict()
        weight_rows.append(dict(family=family, class_weight=label, **r))
weights_df = pd.DataFrame(weight_rows).rename(columns={"model": "configuration"})
weights_df.to_csv(REPORTS / "class_weight_comparison.csv", index=False)
display(weights_df[["family", "class_weight", "rare_precision", "rare_recall", "rare_tp", "rare_fp",
                    "pooled_macro_f1", "balanced_accuracy"]].round(4))

for family, _, _ in pairs:
    w = weights_df[(weights_df.family == family) & (weights_df.class_weight == "balanced")].iloc[0]
    u = weights_df[(weights_df.family == family) & (weights_df.class_weight == "none")].iloc[0]
    print(f"{family}: removing the weights changes rare recall {w.rare_recall:.3f} -> {u.rare_recall:.3f}, "
          f"false positives {int(w.rare_fp)} -> {int(u.rare_fp)}, pooled macro-F1 {w.pooled_macro_f1:.4f} -> {u.pooled_macro_f1:.4f}")

,family,class_weight,rare_precision,rare_recall,rare_tp,rare_fp,pooled_macro_f1,balanced_accuracy
0,Decision Tree,balanced,0.0376,0.9070,39.0,997.0,0.7455,0.9366
1,Decision Tree,none,0.6923,0.2093,9.0,4.0,0.8285,0.8008
2,Random Forest,balanced,0.0380,0.8837,38.0,962.0,0.7465,0.9322
3,Random Forest,none,1.0000,0.2558,11.0,0.0,0.8500,0.8125


Decision Tree: removing the weights changes rare recall 0.907 -> 0.209, false positives 997 -> 4, pooled macro-F1 0.7455 -> 0.8285
Random Forest: removing the weights changes rare recall 0.884 -> 0.256, false positives 962 -> 0, pooled macro-F1 0.7465 -> 0.8500


## 5. Where do the missed `reset-both` rows go?
If the missed rows are all predicted as one neighbouring class, the classes overlap in feature space: the problem
is separability, not just rarity.

In [12]:
cm = confusion_matrix(y_train, oof["KNN (deployed)"], labels=CLASS_LABELS)
r = CLASS_LABELS.index("reset-both")
where = pd.Series(cm[r], index=CLASS_LABELS, name="true reset-both predicted as")
display(where.to_frame())

sub = train[train["Action"].isin(["reset-both", "deny"])]
profile = sub.groupby("Action").agg(
    rows=("Action", "size"),
    nat_source_port_zero_share=("NAT Source Port", lambda s: (s == 0).mean()),
    median_bytes=("Bytes", "median"),
    median_packets=("Packets", "median"),
    median_elapsed_sec=("Elapsed Time (sec)", "median"),
).round(4)
profile.to_csv(REPORTS / "reset_both_vs_deny_profile.csv")
display(profile)

top_ports = (train[train["Action"] == "reset-both"]["Destination Port"]
            .value_counts().head(5).rename("reset-both rows"))
display(top_ports.to_frame())

,true reset-both predicted as
allow,0
deny,32
drop,0
reset-both,11


,rows,nat_source_port_zero_share,median_bytes,median_packets,median_elapsed_sec
Action,,,,,
deny,6434,0.9986,66.0,1.0,0.0
reset-both,43,0.7442,146.0,1.0,0.0


,reset-both rows
Destination Port,
3389,11
25773,5
31652,5
34021,5
61248,4


## 6. Decision-rule scan (analysis only)
Multiplying the `reset-both` probability by a weight `w` before the arg-max makes the model more willing to
predict that class. Scanned on out-of-fold probabilities. Not deployed: it would need a custom pipeline wrapper,
and KNN probabilities are not calibrated. A flat or negative result is still a finding.

In [13]:
assert CLASS_LABELS == sorted(CLASS_LABELS)       # predict_proba columns follow sorted class order
proba = oof_predict(deployed_knn(), FULL, method="predict_proba")

scan_rows = []
for w in [1, 2, 3, 5, 8, 12, 20, 40]:
    p = proba.copy(); p[:, CLASS_LABELS.index("reset-both")] *= w
    pred = np.array(CLASS_LABELS)[p.argmax(axis=1)]
    row = rare_row(f"reset-both probability x{w}", pred); row["weight"] = w
    scan_rows.append(row)
scan_df = pd.DataFrame(scan_rows)[["weight", "rare_precision", "rare_recall", "rare_f1", "rare_tp", "rare_fp",
                                "pooled_macro_f1", "balanced_accuracy"]]
scan_df.to_csv(REPORTS / "decision_threshold_scan.csv", index=False)
display(scan_df.round(4))

,weight,rare_precision,rare_recall,rare_f1,rare_tp,rare_fp,pooled_macro_f1,balanced_accuracy
0,1,1.0000,0.2558,0.4074,11,0,0.8500,0.8126
1,2,1.0000,0.2558,0.4074,11,0,0.8500,0.8126
2,3,1.0000,0.2558,0.4074,11,0,0.8500,0.8126
3,5,0.4783,0.2558,0.3333,11,12,0.8312,0.8121
4,8,0.2000,0.3488,0.2542,15,60,0.8106,0.8335
5,12,0.0994,0.3721,0.1569,16,145,0.7846,0.8360
6,20,0.1034,0.4186,0.1659,18,156,0.7867,0.8472
7,40,0.0503,0.6512,0.0933,28,529,0.7610,0.8909


## 7. Enriched feature ablation (same folds)
Same evaluation as Notebook 03, enriched with the count of `reset-both` rows found / missed / falsely predicted.
`delta_vs_feature_ablation_csv` compares with `reports/feature_ablation.csv`; it should be ~0 if both notebooks
use identical folds and parameters. Read together with `reports/feature_overlap_audit.csv` and
`reports/strict_group_cv.csv`. Optional extension: remove `Elapsed Time (sec)` (defined only in this notebook).

In [14]:
SCENARIOS = ["full", "without_nat", "ports_only", "without_redundant_totals"]
if RUN_EXTENSION:
    prep.FEATURE_SETS["without_elapsed_time"] = [c for c in prep.FEATURE_SETS["full"] if c != "Elapsed Time (sec)"]
    SCENARIOS.append("without_elapsed_time")

abl_rows = []
for scn in SCENARIOS:
    cols = get_feature_columns(scn)
    pred = pred_deployed if scn == "full" else oof_predict(deployed_knn(scn), cols)
    mean_f1, std_f1, mean_ba = fold_means(pred)
    cmx = confusion_matrix(y_train, pred, labels=CLASS_LABELS); r = CLASS_LABELS.index("reset-both")
    delta = (mean_f1 - ref.loc[scn, "f1_macro_mean"]) if scn in ref.index else np.nan
    abl_rows.append(dict(feature_set=scn, n_features=len(cols), f1_macro_mean=mean_f1, f1_macro_std=std_f1,
                        balanced_accuracy_mean=mean_ba, rare_tp=int(cmx[r, r]),
                        rare_fn=int(cmx[r].sum() - cmx[r, r]), rare_fp=int(cmx[:, r].sum() - cmx[r, r]),
                        delta_vs_feature_ablation_csv=delta,
                        in_src_feature_sets=scn != "without_elapsed_time"))
    print("done:", scn)

abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv(REPORTS / "expanded_feature_ablation.csv", index=False)
display(abl_df.round(5))

done: full
done: without_nat
done: ports_only
done: without_redundant_totals
done: without_elapsed_time


,feature_set,n_features,f1_macro_mean,f1_macro_std,balanced_accuracy_mean,rare_tp,rare_fn,rare_fp,delta_vs_feature_ablation_csv,in_src_feature_sets
0,full,11,0.84781,0.03777,0.81390,11,32,0,0.0,True
1,without_nat,9,0.84765,0.03784,0.81373,11,32,0,0.0,True
2,ports_only,2,0.68680,0.00634,0.67643,0,43,0,0.0,True
3,without_redundant_totals,9,0.84775,0.03779,0.81382,11,32,0,0.0,True
4,without_elapsed_time,10,0.84625,0.03790,0.81301,11,32,0,NaN,False


## 8. Permutation importance (validation fold only)
Shuffles one column in a validation fold and measures the macro-F1 drop. Computed for the deployed KNN (what
the app will display) and cross-checked with an unweighted Random Forest. Cautions: model-specific, not causal;
`Bytes`/`Bytes Sent`/`Bytes Received` are mathematically related and share importance; high importance does not
prove availability at decision time; fold 0 has only 8–9 `reset-both` rows, so near-zero values are noise.

In [15]:
tr_idx, va_idx = folds[0]

def perm_table(model, label):
    model.fit(X_train.iloc[tr_idx][FULL], y_train.iloc[tr_idx])
    res = permutation_importance(model, X_train.iloc[va_idx][FULL], y_train.iloc[va_idx],
                                scoring="f1_macro", n_repeats=N_REPEATS,
                                random_state=RANDOM_STATE, n_jobs=1)
    out = pd.DataFrame({"model": label, "feature": FULL,
                        "mean_drop": res.importances_mean, "std_drop": res.importances_std})
    return out.sort_values("mean_drop", ascending=False).reset_index(drop=True)

t0 = time.perf_counter()
perm_knn = perm_table(deployed_knn(), "KNN (deployed)")
perm_rf = perm_table(with_weight("Random Forest", None), "Random Forest (unweighted)")
perm_df = pd.concat([perm_knn, perm_rf], ignore_index=True)
perm_df.to_csv(REPORTS / "permutation_importance.csv", index=False)
print(f"computed in {time.perf_counter() - t0:.0f}s")
display(perm_df.pivot(index="feature", columns="model", values="mean_drop")
        .sort_values("KNN (deployed)", ascending=False).round(4))

computed in 193s


model,KNN (deployed),Random Forest (unweighted)
feature,,
Elapsed Time (sec),0.2414,0.1199
Destination Port,0.2264,0.2722
Bytes Sent,0.0714,0.0000
Bytes Received,0.0246,0.0000
Bytes,0.0202,0.0000
NAT Destination Port,0.0172,0.0335
pkts_received,0.0023,0.0000
Packets,0.0021,0.0004
pkts_sent,0.0020,0.0000


## 9. Prediction latency
Deployed pipeline timed on 1 and 100 rows after fitting on the full training set. Absolute values depend on the
machine, so hardware and library versions are stored with the result.

In [16]:
model_latency = deployed_knn().fit(X_train[FULL], y_train)
lat_rows = []
for n in (1, 100):
    batch = X_train[FULL].head(n)
    times = []
    for _ in range(10):
        t0 = time.perf_counter(); model_latency.predict(batch); times.append((time.perf_counter() - t0) * 1000)
    lat_rows.append(dict(rows=n, median_ms=float(np.median(times)), max_ms=float(np.max(times)),
                        python=platform.python_version(), sklearn=sklearn.__version__,
                        processor=platform.processor() or platform.machine()))
lat_df = pd.DataFrame(lat_rows)
lat_df.to_csv(REPORTS / "prediction_latency.csv", index=False)
display(lat_df.round(2))

,rows,median_ms,max_ms,python,sklearn,processor
0,1,31.51,33.6,3.14.4,1.9.1,x86_64
1,100,96.78,154.3,3.14.4,1.9.1,x86_64


## 10. Feature-availability evidence
`docs/docs/feature_availability.md` states the question in general terms. This table adds the measured effect of
each feature group, built from the results above so the numbers cannot drift from the evidence. The availability
column is an **assumption**: the dataset does not document when each field is produced.

In [17]:
A = abl_df.set_index("feature_set")
P = perm_knn.set_index("feature")["mean_drop"]
full_f1 = A.loc["full", "f1_macro_mean"]

def d(scn): return f"{A.loc[scn, 'f1_macro_mean']:.4f} (delta {A.loc[scn, 'f1_macro_mean'] - full_f1:+.4f} vs full {full_f1:.4f})"

avail = pd.DataFrame([
    dict(feature_group="Source Port / Destination Port",
        availability_assumption="Usually known when the connection is opened",
        concern="Not enough alone to represent firewall policy",
        experiment="ports_only ablation",
        observed_effect=f"ports_only macro-F1 {d('ports_only')}; reset-both found {int(A.loc['ports_only','rare_tp'])}/"
                        f"{int(A.loc['ports_only','rare_tp'] + A.loc['ports_only','rare_fn'])}"),
    dict(feature_group="NAT Source Port / NAT Destination Port",
        availability_assumption="May be assigned as a consequence of the decision (uncertain)",
        concern="Possible target leakage (strong association with Action in the EDA)",
        experiment="without_nat ablation + permutation importance",
        observed_effect=f"without_nat macro-F1 {d('without_nat')}; permutation drop NAT Source {P['NAT Source Port']:.4f}, "
                        f"NAT Destination {P['NAT Destination Port']:.4f}"),
    dict(feature_group="Bytes / Packets (totals)",
        availability_assumption="Measured over the flow (not known at the start)",
        concern="Exactly equal to sent + received in this dataset (redundant)",
        experiment="without_redundant_totals ablation",
        observed_effect=f"without_redundant_totals macro-F1 {d('without_redundant_totals')}"),
    dict(feature_group="Bytes Sent / Received, pkts_sent / pkts_received",
        availability_assumption="Measured over the flow (not known at the start)",
        concern="Completed-flow measurements",
        experiment="permutation importance (deployed KNN)",
        observed_effect="permutation drops: " + ", ".join(f"{c} {P[c]:.4f}" for c in
                        ["Bytes Sent", "Bytes Received", "pkts_sent", "pkts_received"])),
    dict(feature_group="Elapsed Time (sec)",
        availability_assumption="Known only when the flow ends",
        concern="Post-decision measurement that may carry the outcome",
        experiment="permutation importance" + (" + without_elapsed_time ablation" if RUN_EXTENSION else ""),
        observed_effect=f"permutation drop {P['Elapsed Time (sec)']:.4f}" +
                        (f"; without_elapsed_time macro-F1 {d('without_elapsed_time')}" if RUN_EXTENSION else "")),
])
avail.to_csv(REPORTS / "feature_availability.csv", index=False)
display(avail)

,feature_group,availability_assumption,concern,experiment,observed_effect
0,Source Port / Destination Port,Usually known when the connection is opened,Not enough alone to represent firewall policy,ports_only ablation,ports_only macro-F1 0.6868 (delta -0.1610 vs f...
1,NAT Source Port / NAT Destination Port,May be assigned as a consequence of the decisi...,Possible target leakage (strong association wi...,without_nat ablation + permutation importance,without_nat macro-F1 0.8477 (delta -0.0002 vs ...
2,Bytes / Packets (totals),Measured over the flow (not known at the start),Exactly equal to sent + received in this datas...,without_redundant_totals ablation,without_redundant_totals macro-F1 0.8477 (delt...
3,"Bytes Sent / Received, pkts_sent / pkts_received",Measured over the flow (not known at the start),Completed-flow measurements,permutation importance (deployed KNN),"permutation drops: Bytes Sent 0.0714, Bytes Re..."
4,Elapsed Time (sec),Known only when the flow ends,Post-decision measurement that may carry the o...,permutation importance + without_elapsed_time ...,permutation drop 0.2414; without_elapsed_time ...


## 11. Should the deployed model change?
Rules: (1) the decision may use training cross-validation only: the test set was already viewed once in
Notebook 03; (2) differences below the fold-to-fold variation are ties (about 0.01 macro-F1 with 8–9 rare rows per
fold); (3) replacing the model after the test set was viewed would require disclosing that any new test claim is
no longer unbiased.

In [19]:
TIE_MARGIN = 0.01
ranked = all_df.sort_values("pooled_macro_f1", ascending=False).reset_index(drop=True)
dep = all_df[all_df["model"] == "KNN (deployed)"].iloc[0]
best = ranked.iloc[0]
margin = best.pooled_macro_f1 - dep.pooled_macro_f1
display(ranked[["model", "pooled_macro_f1", "mean_fold_macro_f1", "rare_precision", "rare_recall",
                "rare_fp", "balanced_accuracy"]].round(4))

if best["model"] == "KNN (deployed)" or margin < TIE_MARGIN:
    ties = ranked[(ranked.pooled_macro_f1 >= dep.pooled_macro_f1 - TIE_MARGIN)]["model"].tolist()
    print(f"DECISION: keep the deployed KNN. Models within {TIE_MARGIN} pooled macro-F1: {ties}")
else:
    print(f"A different model leads by {margin:.4f} pooled macro-F1 ({best['model']}). "
        "Discuss with the team: re-export would require disclosing that the test set was already seen.")

,model,pooled_macro_f1,mean_fold_macro_f1,rare_precision,rare_recall,rare_fp,balanced_accuracy
0,Random Forest (unweighted),0.8500,0.8479,1.0000,0.2558,0,0.8125
1,KNN (deployed),0.8500,0.8478,1.0000,0.2558,0,0.8126
2,Decision Tree (unweighted),0.8285,0.8221,0.6923,0.2093,4,0.8008
3,Linear SVM (balanced),0.8091,0.8113,0.2391,0.2558,35,0.8112
4,Random Forest (balanced_subsample),0.7465,0.7464,0.0380,0.8837,962,0.9322
5,Decision Tree (balanced),0.7455,0.7455,0.0376,0.9070,997,0.9366
6,Gaussian Naive Bayes,0.7074,0.7074,0.0184,1.0000,2293,0.9147


DECISION: keep the deployed KNN. Models within 0.01 pooled macro-F1: ['Random Forest (unweighted)', 'KNN (deployed)']


## 12. Optional: update `models/metrics.json` for the Streamlit page
Changes only two keys: `cv_feature_ablation` (the page shows what is in the list) and `feature_importances`
(list of `{"feature", "importance"}`, format of `tests/fixtures/sample_metrics_FAKE.json`). Disabled by default.
The extension scenario is not in `src/preprocessing.py`, so it is not written. Importances come from the deployed
KNN on one validation fold.

In [20]:
if UPDATE_METRICS_JSON:
    path = ROOT / "models" / "metrics.json"
    metrics = json.loads(path.read_text(encoding="utf-8"))

    keep = abl_df[abl_df["in_src_feature_sets"]]
    metrics["cv_feature_ablation"] = [
        dict(feature_set=r.feature_set, n_features=int(r.n_features),
            f1_macro_mean=float(r.f1_macro_mean), f1_macro_std=float(r.f1_macro_std),
            balanced_accuracy_mean=float(r.balanced_accuracy_mean),
            rare_tp=int(r.rare_tp), rare_fn=int(r.rare_fn), rare_fp=int(r.rare_fp))
        for r in keep.itertuples()
    ]
    metrics["feature_importances"] = [
        dict(feature=r.feature, importance=round(float(r.mean_drop), 6))
        for r in perm_knn.itertuples()
    ]
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")   # same formatting as the existing file
    print("updated:", path.relative_to(ROOT), "| ablation rows:", len(metrics["cv_feature_ablation"]))
else:
    print("UPDATE_METRICS_JSON is False: models/metrics.json was not modified.")

UPDATE_METRICS_JSON is False: models/metrics.json was not modified.


In [21]:
dep = all_df.set_index("model").loc["KNN (deployed)"]
n_rare = int((y_train == "reset-both").sum())
print(f"reset-both rows in training set: {n_rare}")
print(f"deployed KNN (out-of-fold): precision {dep.rare_precision:.3f}, recall {dep.rare_recall:.3f}, "
    f"{int(dep.rare_tp)} found, {int(dep.rare_fn)} missed ({int(dep.misses_predicted_deny)} predicted as deny), "
    f"{int(dep.rare_fp)} false alarms")
hi = all_df[(all_df.rare_recall > 0.8)]
print("models with rare recall > 0.8:", {r.model: int(r.rare_fp) for r in hi.itertuples()}, "(value = false positives)")
print(f"ablation macro-F1 range among full/without_nat/without_redundant_totals: "
    f"{A.loc[['full','without_nat','without_redundant_totals'], 'f1_macro_mean'].min():.4f} - "
    f"{A.loc[['full','without_nat','without_redundant_totals'], 'f1_macro_mean'].max():.4f}; "
    f"fold std ~ {A.loc['full','f1_macro_std']:.4f}")
print(f"ports_only macro-F1: {A.loc['ports_only','f1_macro_mean']:.4f}, rare found: {int(A.loc['ports_only','rare_tp'])}")
print("top permutation features (KNN):", perm_knn.head(3)[["feature", "mean_drop"]].round(4).values.tolist())

reset-both rows in training set: 43
deployed KNN (out-of-fold): precision 1.000, recall 0.256, 11 found, 32 missed (32 predicted as deny), 0 false alarms
models with rare recall > 0.8: {'Decision Tree (balanced)': 997, 'Random Forest (balanced_subsample)': 962, 'Gaussian Naive Bayes': 2293} (value = false positives)
ablation macro-F1 range among full/without_nat/without_redundant_totals: 0.8477 - 0.8478; fold std ~ 0.0378
ports_only macro-F1: 0.6868, rare found: 0
top permutation features (KNN): [['Elapsed Time (sec)', 0.2414], ['Destination Port', 0.2264], ['Bytes Sent', 0.0714]]


## Conclusions

**1. Accuracy hides the rare class.** The dataset is dominated by a few classes, so a model can be almost
always right while finding few `reset-both` rows. Macro-F1 and per-class precision/recall are the relevant measures.

**2. The models fall into two families.** High-precision models (the deployed KNN, the unweighted forest) miss most
`reset-both` rows but almost never raise a false alarm. Class-weighted models find most of them at the cost of a
large number of false positives. Neither is better in the abstract: the choice depends on the cost of a false alarm
versus a missed event, which this dataset cannot tell us.

**3. The missed rows are not random.** They are predicted as `deny`: the two classes overlap in feature space, so
the limit is separability as well as rarity.

**4. Changing the decision rule does not remove the trade-off.** Scaling the `reset-both` probability recovers some
rows only by sharply reducing precision (analysis only; not deployed; KNN probabilities are not calibrated).

**5. Feature robustness.** Removing the NAT ports or the redundant totals leaves macro-F1 essentially unchanged
(differences are inside the fold-to-fold variation). Using ports alone loses a lot of performance and cannot find
`reset-both`, so completed-flow measurements carry much of the signal. Permutation importance ranks
`Elapsed Time (sec)` and `Destination Port` highest for the deployed KNN, but removing the duration costs little
macro-F1, because other features can substitute for it: importance measures dependence of this model, not necessity
of the feature. `NAT Destination Port` shows a non-negligible but unstable importance (large standard deviation).
These measurements (and possibly NAT) may not exist at decision time, so the results describe retrospective
classification of logged flows, not demonstrated prediction before a firewall decision.

**6. Limits.** There are only 43 training and 11 test `reset-both` rows, so every rare-class figure is unstable (one
row changes recall by several points). One firewall policy was observed. Importance scores are not causal, and were
computed on a single fold and a single machine.

**7. Model decision.** See section 11: the deployed model is kept unless the evidence shows a difference larger than
the noise level, because the held-out test set was already viewed once.

## Summary
Adds `notebooks/04_ml_improvement_experiments.ipynb` and the `reports/` files it generates. Notebook 03 is unchanged.

## What was done
- Out-of-fold rare-class (`reset-both`) comparison of the six classifiers, using the deployed KNN hyperparameters
- Class-weight comparison, error profile (reset-both vs deny), decision-rule scan (analysis only, not deployed)
- Enriched four-scenario feature ablation (+ one notebook-local extension) with rare-class counts
- Permutation importance on a validation fold, prediction latency, feature-availability evidence table
- Test for the `without_redundant_totals` feature set

## Methodology guarantees
- Training data only: `test.csv` is never loaded; no model selection uses the held-out set
- Same group-aware folds as Notebook 03 (hash of the full feature row)
- All numbers in reports come from executed code (no hand-typed values)

## Verification
- `python -m pytest -q` : all tests pass
- Notebook run top to bottom (Restart & Run All) without errors

## Known limits
43 training / 11 test `reset-both` rows; single firewall policy; retrospective classification only.